# Day 2 · 버그 사냥 — 정답 없이 버그를 잡을 수 있나?
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kehdgus9434/sonic-vibe-coding-lab/blob/main/notebooks/day2_bughunt.ipynb)

**오늘의 결과물**: 내 불변식 테스트 → 그 테스트로 고친 파이프라인 → (마지막에만) 정답 비교 → 40초 완주 영상 + 제출 카드

실제 연구에는 정답 코드가 없습니다. 그래서 오늘은 **테스트를 먼저 만들고, 그 테스트만으로 버그를 고친 뒤, 마지막에 한 번 정답과 비교**합니다.

| 시간 | 내용 |
|---|---|
| 0:00–0:10 | 환경 준비 · 증상 관찰 |
| 0:10–0:45 | Mission 2-1 불변식 테스트 만들기 (정답 비교 없음) |
| 0:45–1:20 | Mission 2-2 내 테스트로 버그 고치기 (정답 비교 없음) |
| 1:20–1:35 | Mission 2-3 최종 판정 — 정답 비교(오라클) |
| 1:35–1:50 | Mission 2-4 40초 완주와 영상 |
| 1:50–2:00 | 📮 제출 카드 · (빨리 끝나면 🚀 심화) |


**AI 사용 규칙 (이 과정은 AI 에이전트 사용을 전제로 합니다)**
- 코드는 어떤 도구로 써도 됩니다: Colab의 Gemini, 웹 채팅(ChatGPT·Claude·Gemini), 웹 코딩 에이전트(Claude Code on the web, Codex) — 설치 없이 브라우저만으로 (`docs/00_setup.md`).
- 대신 **평가는 코드가 아니라 당신의 판단**입니다: 🔮 예측(실행 전), 🧾 검증 기록(체크 후), 마지막 📮 제출 카드는 **본인이 직접** 씁니다.
- `sonic_lab/reference.py`·`check.py`는 정답입니다. 에이전트에게도 읽지 말라고 지시하세요(`AGENTS.md`에 규칙이 있음).

- 오늘은 추가로: `check.bughunt`·`check.invariant_tests`(정답 비교)는 **Mission 2-3 전에는 부르지 않습니다.** 호출 횟수가 기록되어 제출 카드에 함께 남습니다.

In [ ]:
#@title 0. 실습 환경 준비 — 처음 한 번 실행 (약 1–2분)
REPO = "https://github.com/kehdgus9434/sonic-vibe-coding-lab"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
# 웹 코딩 에이전트 사용자: PR 댓글의 "Colab에서 내 브랜치로 열기" 링크를 쓰면 이 두 값이 자동으로 채워집니다
import os, sys, subprocess
from pathlib import Path
if Path("/content").exists():                              # Google Colab
    LAB_DIR = Path("/content/sonic-vibe-coding-lab")
    if not LAB_DIR.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO, str(LAB_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(LAB_DIR / "requirements-colab.txt")], check=True)
else:                                                      # 로컬 Jupyter / VS Code (저장소 안에서 실행)
    LAB_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(LAB_DIR)
sys.path.insert(0, str(LAB_DIR))
import numpy as np
import sonic_lab
from sonic_lab import check, reference as ref, render, analysis
lab = sonic_lab.start()

In [ ]:
#@title 🔄 (웹 에이전트 사용자) 에이전트가 Fork에 push한 최신 코드 받기 — 필요할 때만
r = subprocess.run(["git", "-C", str(LAB_DIR), "pull", "--ff-only", "origin", BRANCH], capture_output=True, text=True)
print(r.stdout or r.stderr)
for name in [m for m in sys.modules if m.startswith("my_work")]:
    del sys.modules[name]                                  # 다음 import에서 새 코드를 읽도록
print("완료 — my_work 코드를 쓰는 ✍️ 셀을 다시 실행하세요")

In [ ]:
# 버그 사냥 대상 불러오기 — 파일을 고칠 때마다 이 셀부터 다시 실행
import importlib.util
def load_buggy(path="exercises/buggy_pipeline.py"):
    spec = importlib.util.spec_from_file_location("buggy_pipeline", path)
    module = importlib.util.module_from_spec(spec); spec.loader.exec_module(module)
    return module.Pipeline()
pipeline = load_buggy()

## 증상 관찰 (5분)
AI가 작성한 파이프라인(`exercises/buggy_pipeline.py`)은 오류 없이 실행되고, 주석도 그럴듯합니다. 파일을 한 번 훑어본 뒤 예측하세요.

> 🔮 **예측 — 아래 셀을 실행하기 전에 직접 적으세요 (AI에게 맡기지 않기)**
> 이 파이프라인으로 G1은 몇 프레임(50 프레임 = 1초)쯤 버틸까요?
>
> 내 예측: ______

In [ ]:
bug_summary, bug_arrays = ref.run_physics(lab.sim, lab.policy, lab.motion, mode=2, max_frames=250, pipeline=pipeline)
print({k: bug_summary[k] for k in ["simulated_frames", "requested_frames", "stop_reason"]})
render.show(render.skeleton_video(lab, bug_arrays, "outputs/day2_buggy.mp4", title="buggy pipeline", seconds=5))

---
## Mission 2-1 · 불변식 테스트 만들기 — 정답 비교 없음 (35분)
> 🧭 아래 설명 + `docs/01_contract.md` + 함수 시그니처를 AI에게 주고 ✍️ 셀을 채운 뒤 ✅ 셀로 확인합니다.

- **목표**: `pipeline`의 세 함수(`observation`, `decoder_obs`, `action_to_target`)가 계약서를 지키는지 확인하는 `assert`를 **4개 이상** 만든다.
- **금지**: `ref.*`, `lab.policy.decoder_obs` 같은 정답 함수의 출력과 비교하기. 쓸 수 있는 것은 `lab.sim`(상태를 직접 설정), `lab.contract`(기본각·스케일·관절 순서), 계약서뿐입니다.
- **질문에서 시작하세요**
  - 로봇이 기본 자세로 똑바로 서 있으면 관측 `q`는? `gravity`는? 기저를 x축으로 90° 기울이면?
  - 이력에 **프레임 번호를 값으로** 넣으면 Decoder 입력 안에서 어떤 순서로 나타나야 할까?
  - `action = 0`이면 목표 관절각은? 한 관절만 1이면 **어떤 이름의** 관절이 움직여야 할까? (`lab.sim.model.joint(id).name`, `lab.sim.jids`)
- **완료 기준**: 아래 실행 셀에서 내 테스트가 **현재 버그 파이프라인에서 실패**한다(= 버그를 하나 이상 잡았다).

In [ ]:
# ✍️ Mission 2-1
import mujoco
from scipy.spatial.transform import Rotation as R

def invariant_tests(pipeline, lab):
    """정답 구현 없이 버그를 잡는 테스트. 문제가 있으면 AssertionError (메시지에 무엇이 틀렸는지)."""
    sim, c = lab.sim, lab.contract
    raise NotImplementedError

# from my_work.day2 import *

In [ ]:
# ▶ 내 테스트 실행 (정답 비교 아님) — 첫 번째로 실패한 assert를 보여 줍니다
def run_my_tests(p):
    try:
        invariant_tests(p, lab); print("내 테스트 모두 통과")
    except AssertionError as error:
        print("❗ 내 테스트 실패:", error)
    finally:
        lab.sim.reset(lab.motion)
run_my_tests(load_buggy())

---
## Mission 2-2 · 내 테스트로 버그 고치기 — 정답 비교 없음 (35분)

- **대상**: `exercises/buggy_pipeline.py` (Colab 왼쪽 📁 → `sonic-vibe-coding-lab/exercises/`에서 더블클릭해 편집, 또는 에이전트가 직접 수정)
- 버그는 **4개**입니다. 내 테스트가 실패를 보여 주면 → 원인 후보를 계약서 조항과 연결 → 고침 → 파일 맨 위 `BUGLOG`에 "무엇을, 어떤 근거로" 기록.
- 테스트가 모두 통과하는데도 의심되면 **테스트를 늘리세요.** 짧은 폐루프 실행도 근거가 됩니다(아래 셀).
- ⚠️ Colab 런타임이 끊기면 파일 수정도 사라집니다. 고친 코드를 수시로 아래 메모 셀에 복사해 두세요.

In [ ]:
# ▶ 고칠 때마다: 내 테스트 + 5초 폐루프 (정답 비교 아님)
pipeline = load_buggy()
run_my_tests(pipeline)
s, _ = ref.run_physics(lab.sim, lab.policy, lab.motion, mode=2, max_frames=250, pipeline=pipeline)
print("폐루프:", s["simulated_frames"], "/ 250 프레임,", s["stop_reason"])

```python
# 📋 고친 코드 백업용 메모 (런타임이 끊기면 여기서 복구)
```

> 🔮 **예측 — 아래 셀을 실행하기 전에 직접 적으세요 (AI에게 맡기지 않기)**
> 버그를 다 고쳤다고 생각하나요? 정답과 비교하면 4개 중 몇 개를 맞혔을까요? 내 테스트가 놓쳤을 것 같은 버그가 있다면?
>
> 내 예측: ______

---
## Mission 2-3 · 최종 판정 — 정답 비교(오라클) (15분)
이제 처음으로 정답과 비교합니다. 두 가지를 봅니다.
1. `check.bughunt`: 내가 고친 파이프라인이 정답과 같은가?
2. `check.invariant_tests`: 내 테스트가 버그 4유형을 **각각 따로** 넣었을 때 몇 개를 잡는가?

In [ ]:
# ✅ 최종 판정 (호출 횟수 기록됨)
_ = check.bughunt(lab, load_buggy(), frames=250)
check.invariant_tests(lab, invariant_tests)

> 🧾 **검증 기록 — 직접 적으세요**
> 넘어지지 않는데도 남아 있던 버그가 있었나요? 그 버그는 폐루프 결과(완주 여부)로는 왜 드러나지 않았을까요? 내 테스트가 놓친 버그 유형과 그 이유는?
>
> 근거: ______

---
## Mission 2-4 · 40초 완주와 영상 (15분)
고친 파이프라인으로 공식 샘플 전체(2,002프레임)를 실행하고 참조 구현과 비교합니다.

> 🔮 **예측 — 아래 셀을 실행하기 전에 직접 적으세요 (AI에게 맡기지 않기)**
> 내 파이프라인과 참조 구현의 관절 오차(°)는 완전히 같을까요, 조금 다를까요? 왜?
>
> 내 예측: ______

In [ ]:
pipeline = load_buggy()
fixed = ref.run_physics(lab.sim, lab.policy, lab.motion, mode=2, pipeline=pipeline, verbose=True)
reference = ref.run_physics(lab.sim, lab.policy, lab.motion, mode=2, verbose=True)
analysis.table({"내 파이프라인": fixed, "참조 구현": reference})

In [ ]:
# 영상 (GPU 런타임이면 EGL, CPU면 OSMesa 소프트웨어 렌더링 · 6초)
video = render.mesh_video(lab, fixed[1], "outputs/day2_fixed.mp4", label="fixed pipeline / mode 2", seconds=6)
render.show(video or render.skeleton_video(lab, fixed[1], "outputs/day2_fixed_skeleton.mp4", title="fixed", seconds=6))

---
## 📮 제출 카드 (10분)
정답 비교 호출 횟수도 함께 출력됩니다. 횟수가 많다고 감점이 아니라, 판단 과정을 돌아보는 자료입니다.

In [ ]:
# 📮 오늘의 제출 카드 — 본인이 직접 작성 (에이전트 작성 금지)
card = {
    "예측과 실제가 달랐던 것": "",
    "AI가 틀렸던 순간과 알아챈 방법": "",
    "가장 믿을 만한 ✅와 그 근거": "",
    "정답 비교 없이 내가 확인한 것": "",
}
check.reflection(card)

---
## 🚀 심화 미션 (정답 없음)

**A. 버그 민감도 지도** — 고친 파이프라인에 버그를 **하나씩만** 다시 넣은 변형 4개를 직접 만들어, 각각 몇 프레임 버티는지와 관절 오차를 표로 만드세요. 어떤 버그가 가장 '조용한' 버그인가요? 조용한 버그를 잡으려면 어떤 지표를 봐야 할까요?

**B. 무작위 상태 테스트** — 기저 회전·관절각·속도를 무작위로 수백 번 바꿔 가며 불변식을 검사하는 property-based 테스트로 확장하세요. 고정된 몇 개 상태만 쓰는 테스트가 놓칠 수 있는 버그를 예로 들어 보세요.

**C. 이력 초기화** — 시작할 때 이력 10프레임을 0으로 채우면 어떻게 될까요? 계약서 4절의 초기화 규칙과 비교하고, 그 차이가 처음 1초 동안 결과에 어떤 영향을 주는지 측정하세요.

In [ ]:
# 🚀 심화 작업 공간